# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and analyze the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library, referencing all data elements by their Croissant `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load Croissant metadata and available records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Explore available record sets (`cr:RecordSet`), their fields (`cr:Field`), and columns. All items are referenced by their Croissant `@id`.


In [ ]:
# List all available record sets and their IDs
print("Available record sets (by @id):")
for record_set in metadata.record_sets:
    print(f"- @id: {record_set.id}, name: {record_set.name}")

# View fields and columns for each record set
for record_set in metadata.record_sets:
    print(f"\nRecord set @id: {record_set.id} (name: {record_set.name})")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - @id: {field.id}, name: {field.name}, dataType: {field.data_type}")
        if hasattr(field, 'column') and field.column:
            if isinstance(field.column, list):
                for col in field.column:
                    print(f"        Column @id: {col.id}, name: {col.name}")
            else:
                col = field.column
                print(f"        Column @id: {col.id}, name: {col.name}")

## 3. Data Extraction
Load records for one or more record sets into pandas DataFrames, referencing everything by `@id`.
Choose the main record set (look for @id and name in the above output).

In [ ]:
# Gather all record set @id values
record_sets_ids = [rs.id for rs in metadata.record_sets]
print("Record set IDs for extraction:", record_sets_ids)

dataframes = {}
for rs_id in record_sets_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records from record set {rs_id}")

# Inspect columns from the first record set
primary_rs_id = record_sets_ids[0]
print(f"\nColumns in primary record set ({primary_rs_id}):\n", dataframes[primary_rs_id].columns.tolist())
dataframes[primary_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Demonstrate basic analysis on numeric or categorical fields by their `@id`. You may filter, normalize, or group if fields exist in the main record set.


In [ ]:
# Find a numeric field in the primary record set
df = dataframes[primary_rs_id]

# Automatically find a float/integer column (by minimal type checking)
numeric_candidates = df.select_dtypes(include=['float', 'int']).columns.tolist()
if not numeric_candidates:
    # Attempt to convert numeric columns if not auto-detected
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            pass
    numeric_candidates = df.select_dtypes(include=['float', 'int']).columns.tolist()

if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
    print(f"Numeric field chosen for analysis: {numeric_field_id}")
    threshold = df[numeric_field_id].quantile(0.75)  # upper quartile
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Choose a grouping field if any non-numeric columns exist
    group_candidates = [c for c in df.columns if df[c].dtype == 'object']
    if group_candidates:
        group_field_id = group_candidates[0]
        print(f"\nGrouping by: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(grouped_df.head())
    else:
        group_field_id = None
else:
    numeric_field_id = None
    group_field_id = None
    print("No numeric fields found for analysis.")

## 5. Visualization
Visualize numeric distributions or groups. The chosen fields are referenced by their Croissant `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()
    
    if group_field_id:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion
- This notebook demonstrated loading, exploring, and analyzing the FAIR² mass spectrometry dataset using `mlcroissant` by referencing each record set and field by `@id`.
- You can extend the template by choosing additional fields or record sets for further processing and carrying out domain-specific analyses.

**Note:** All dataset components were referenced by their Croissant JSON-LD `@id` for clarity and reproducibility.